# 02 · HTTP API 与 httptest

目标：只用标准库完成 JSON 边界、状态码映射和无需真实端口的 handler 测试。对应文档：`02_project_structure/2.3-api-service-anatomy.md`。

## 从 Express/Supertest 迁移

`http.Handler` 对应路由处理边界，`httptest.NewRecorder` 类似无真实端口的请求测试。Go 倾向让 JSON 解码、业务调用和错误映射保持显式。

In [ ]:
import (
    "bytes"
    "encoding/json"
    "fmt"
    "net/http"
    "net/http/httptest"
    "strings"
)

type CreateResourceRequest struct {
    Title string `json:"title"`
}

func resourceHandler() http.Handler {
    return http.HandlerFunc(func(w http.ResponseWriter, r *http.Request) {
        if r.Method != http.MethodPost {
            w.Header().Set("Allow", http.MethodPost)
            http.Error(w, "method not allowed", http.StatusMethodNotAllowed)
            return
        }

        var input CreateResourceRequest
        decoder := json.NewDecoder(r.Body)
        decoder.DisallowUnknownFields()
        if err := decoder.Decode(&input); err != nil || strings.TrimSpace(input.Title) == "" {
            http.Error(w, "invalid request", http.StatusBadRequest)
            return
        }

        w.Header().Set("Content-Type", "application/json")
        w.WriteHeader(http.StatusCreated)
        _ = json.NewEncoder(w).Encode(map[string]string{
            "id": "res-1", "title": strings.TrimSpace(input.Title),
        })
    })
}

## 实验一：成功边界

先预测状态码、Content-Type 和返回标题。注意 handler 在边界执行 trim，而不是要求调用方提前清洗输入。

In [ ]:
%%
validRequest := httptest.NewRequest(
    http.MethodPost,
    "/resources",
    bytes.NewBufferString(`{"title":" Learn Go "}`),
)
validResponse := httptest.NewRecorder()
resourceHandler().ServeHTTP(validResponse, validRequest)

fmt.Println("status:", validResponse.Code)
fmt.Println("content-type:", validResponse.Header().Get("Content-Type"))
fmt.Println("body:", validResponse.Body.String())

## 实验二：失败矩阵

空标题、未知字段和非法 JSON 都属于 400，但失败原因不同。稳定协议可以统一公开错误，同时在服务端保留足够诊断信息。

In [ ]:
%%
badBodies := []string{
    `{"title":""}`,
    `{"title":"Go","unknown":true}`,
    `not-json`,
}

for _, body := range badBodies {
    request := httptest.NewRequest(http.MethodPost, "/resources", bytes.NewBufferString(body))
    response := httptest.NewRecorder()
    resourceHandler().ServeHTTP(response, request)
    fmt.Printf("body=%-35s status=%d response=%q\n", body, response.Code, response.Body.String())
}

## 验证真实分层服务

下面直接运行 `02_project_structure/api_service` 的端到端 handler 测试，覆盖创建、列表、完成和进度查询。

In [ ]:
!*cd "$GO_LEARNING_API_SERVICE" && go test ./internal/httpapi -run 'TestHandlerCreatesListsCompletesAndReportsProgress|TestHandlerRejectsInvalidCreateRequests' -v

## 动手挑战

1. 增加 GET 方法并正确设置 `Allow`。
2. 拒绝一个 JSON 对象后的多余 token。
3. 加入请求体大小限制，思考应返回 400 还是 413。

## 复盘

- transport、service、repository 各自负责什么？
- 为什么 handler 测试不需要监听真实端口？
- 哪些内部错误绝不能直接返回客户端？